# Iterative Temporal Parent Discovery

Dev | Jacqueline Maasch | August 2026

In [1]:
# General imports.
import pandas as pd
import numpy as np
#import pickle
import matplotlib.pyplot as plt
import time
from string import ascii_uppercase as alphabet
import scipy.stats as st

# Time series causal discovery.
import tigramite
from tigramite import data_processing as pp
from tigramite.toymodels import structural_causal_processes as toys
from tigramite import plotting as tp
from tigramite.pcmci import PCMCI # also for PCMCI+
from tigramite.independence_tests.gsquared import Gsquared # univariate discrete/categorical vars
from tigramite.independence_tests.cmisymb import CMIsymb # multivariate discrete/categorical vars (permutation-based)
from tigramite.independence_tests.parcorr import ParCorr # univariate, continuous variables with linear dependencies and Gaussian noise

# VLCD.
from vlcd import PCMCIConfig, pcmci

# Vanilla causal discovery.
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.utils.cit import CIT

# Custom discovery algorithms.
from padl_itpd import PaDL
from itpd import ITPD, Utils
# Naive implementation.
from padl import PaDL
from itpd_naive import ITPDNaive

# Metrics.
#from dodiscover.metrics import structure_hamming_dist as SHD
from cdt.metrics import precision_recall
from cdt.metrics import SHD
from cdt.metrics import SID

# Graphical modeling.
import networkx as nx

No GPU automatically detected. Setting SETTINGS.GPU to 0, and SETTINGS.NJOBS to cpu_count.


# ITPD

## Generate random time series

In [2]:
# Get random data.
u = Utils()
dfs = []
graphs = []
adj_matrices = []
p2c_list = []
c2p_list = []
M = 1_0
T = 10
N = 10
max_children = 4
plot = False
iters = 10 # total random datasets to generate.

for i in range(iters):
    df, p2c, c2p, G, true_adj = u.get_data(M = M, 
                                           T = T,
                                           N = N, 
                                           max_children = max_children,
                                           plot = plot)
    dfs.append(df)
    graphs.append(G)
    adj_matrices.append(true_adj)
    p2c_list.append(p2c)
    c2p_list.append(c2p)

In [3]:
dfs[0]

,A_0,A_1,A_2,A_3,A_4,A_5,A_6,A_7,A_8,A_9,...,J_0,J_1,J_2,J_3,J_4,J_5,J_6,J_7,J_8,J_9
0,-1.205239,-0.257437,-1.019311,-0.461069,0.617614,-0.133112,3.114470,4.661338,-1.505167,2.881409,...,0.602825,0.138846,-0.116797,0.486724,-1.832897,-4.297543,-10.882282,-1.129978,-31.680052,28.946184
1,1.699562,2.250073,-1.231594,-2.530138,0.709799,-0.344322,-2.945408,-4.201243,7.529753,0.419049,...,-0.333097,-2.837574,-5.345496,4.595881,-2.330109,-1.224881,2.029060,-2.141811,-8.893635,11.177568
2,0.049435,-1.735105,2.974816,2.039814,-0.281986,2.599314,1.956662,2.325009,-4.899774,-1.545265,...,-0.545327,-1.486008,-2.933613,3.764785,-6.445972,-7.578329,-3.870554,2.871630,13.511657,-40.042996
3,-1.660687,2.008343,-1.156997,-2.061670,-0.898177,0.590151,-0.543067,-2.677687,3.412808,1.070000,...,0.466640,1.165084,1.533520,-2.383501,5.021819,4.579957,8.276047,-6.953265,-17.896265,30.547052
4,1.183974,-1.515509,2.602307,-0.379808,-1.928758,-2.361958,3.568198,5.321376,-7.300680,5.071748,...,-1.716825,-1.878999,-2.479029,1.225651,-4.229008,-9.100806,-13.416660,7.879465,-13.743361,60.141341
5,-1.629972,2.372103,-1.978693,-2.460870,4.290376,4.897418,-2.792270,-4.125733,2.212954,-2.213558,...,-0.214562,-0.270949,-1.041342,1.218919,-1.756003,-3.034063,-2.914597,1.109850,-19.367445,64.975116
6,2.288001,-1.974104,1.355276,2.409098,-1.946148,-2.260752,1.262440,2.806188,-6.673140,-2.741001,...,-0.632132,-1.153297,-1.903522,1.958587,-3.647731,-4.718485,-5.819637,7.883245,-6.461938,40.838972
7,-2.031591,2.452932,-1.990799,-1.577381,7.216450,8.812356,-8.129150,-13.258916,13.239120,-8.195137,...,1.326492,1.551159,2.017070,-1.828861,1.578070,1.747658,5.661035,-3.918696,-13.128661,31.331146
8,-0.568818,0.782013,-1.665488,-0.241943,-2.102291,-2.841133,2.716711,4.183623,1.481285,4.875374,...,-0.280923,-1.767290,-3.345064,2.031881,0.195716,3.251194,6.143138,-8.944977,-25.114512,29.551400
9,1.050523,-1.264315,0.383995,1.534929,-5.848385,-3.238303,0.048314,-0.651746,3.528856,9.718894,...,0.014819,0.875069,1.082372,-1.522693,-0.506020,1.932042,0.848758,-6.924544,-11.432298,-55.328603


In [4]:
p2c_list[0]

{'A_0': ['A_1', 'I_9', 'F_9', 'H_5'],
 'A_1': ['A_2', 'F_2', 'H_4'],
 'A_2': ['A_3', 'H_4'],
 'A_3': ['A_4', 'C_4'],
 'A_4': ['A_5', 'F_7', 'D_9', 'C_8', 'A_9'],
 'A_5': ['A_6', 'C_8', 'F_6', 'G_6', 'F_7'],
 'A_6': ['A_7', 'C_9'],
 'A_7': ['A_8', 'D_8', 'G_8', 'I_8', 'A_9'],
 'A_8': ['A_9', 'E_9'],
 'A_9': [],
 'B_0': ['B_1', 'H_6', 'F_2', 'B_3', 'H_1'],
 'B_1': ['B_2', 'A_4', 'H_6', 'F_8', 'F_7'],
 'B_2': ['B_3', 'C_4', 'I_5', 'J_6', 'C_4'],
 'B_3': ['B_4'],
 'B_4': ['B_5'],
 'B_5': ['B_6', 'J_9', 'G_7'],
 'B_6': ['B_7', 'B_8', 'D_8', 'H_8', 'A_9'],
 'B_7': ['B_8', 'E_9', 'C_8', 'A_8'],
 'B_8': ['B_9', 'D_9', 'C_9'],
 'B_9': [],
 'C_0': ['C_1', 'I_9', 'H_3', 'B_3'],
 'C_1': ['C_2'],
 'C_2': ['C_3', 'H_9', 'B_4', 'B_3'],
 'C_3': ['C_4'],
 'C_4': ['C_5', 'H_7', 'F_7', 'B_6', 'E_9'],
 'C_5': ['C_6', 'D_6', 'I_6', 'F_8'],
 'C_6': ['C_7'],
 'C_7': ['C_8', 'H_8', 'J_8', 'A_9'],
 'C_8': ['C_9', 'J_9'],
 'C_9': [],
 'D_0': ['D_1'],
 'D_1': ['D_2', 'C_4', 'F_3', 'C_9', 'A_4'],
 'D_2': ['D_3', 

## Oracle

### ITPD

In [14]:
test = "oracle"
alpha = 0.01
tau_max = None
pairs = None
var_names = list(alphabet[:N])
ci_to_float = lambda x : (round(float(x[0]),3), round(float(x[1]),3))

accuracies = []
auprs = []
shds = []
total_tests = []
cond_sizes = []
itpd_oracle_times = []

for i in range(iters):
    start = time.time()
    it = ITPD(dfs[i],
              var_names = var_names,
              pairs = pairs,
              independence_test = test,
              alpha = alpha,
              tau_max = tau_max,
              true_adj = adj_matrices[i])
    accuracy, aupr, shd = it.itpd()
    itpd_oracle_times.append(time.time() - start)
    
    total_tests.append(np.sum(it.total_tests))
    cond_sizes_flat = [item for sublist in it.conditioning_set_sizes for item in sublist]
    cond_sizes += cond_sizes_flat
    
    accuracies.append(accuracy)
    auprs.append(aupr)
    shds.append(shd)


# Get 95% confidence intervals.
# Assuming t distribution.
time_ci = st.t.interval(0.95, 
                        len(itpd_oracle_times)-1, 
                        loc = np.mean(itpd_oracle_times), 
                        scale = st.sem(itpd_oracle_times))
tests_ci = st.t.interval(0.95, 
                        len(total_tests)-1, 
                        loc = np.mean(total_tests), 
                        scale = st.sem(total_tests))
cond_ci = st.t.interval(0.95, 
                        len(cond_sizes)-1, 
                        loc = np.mean(cond_sizes), 
                        scale = st.sem(cond_sizes))
auprs_ci = st.t.interval(0.95, 
                        len(auprs)-1, 
                        loc = np.mean(auprs), 
                        scale = st.sem(auprs))
shds_ci = st.t.interval(0.95, 
                        len(shds)-1, 
                        loc = np.mean(shds), 
                        scale = st.sem(shds))

print()
print(f"Mean runtime     : {round(np.mean(itpd_oracle_times),4)}s {ci_to_float(time_ci)}")
print(f"Mean total tests : {round(np.mean(total_tests),2)} {ci_to_float(tests_ci)}")
print(f"Mean cond. set : {round(np.mean(cond_sizes),2)} {ci_to_float(cond_ci)}")
print()
print(f"Mean AUPR        : {round(np.mean(auprs)*100,2)} {ci_to_float(auprs_ci)}")
print(f"Mean SHD         : {round(np.mean(shds),2)} {ci_to_float(shds_ci)}")

ITPD complete in 3.0247 seconds.
ITPD complete in 2.7804 seconds.
ITPD complete in 2.9995 seconds.
ITPD complete in 2.8522 seconds.
ITPD complete in 3.0874 seconds.
ITPD complete in 3.0653 seconds.
ITPD complete in 2.9508 seconds.
ITPD complete in 3.1172 seconds.
ITPD complete in 3.2628 seconds.
ITPD complete in 2.9637 seconds.

Mean runtime     : 3.0119s (2.914, 3.11)
Mean total tests : 16330.4 (16004.655, 16656.145)
Mean cond. set : 66.68 (66.403, 66.957)

Mean AUPR        : 100.0 (nan, nan)
Mean SHD         : 0.0 (nan, nan)


invalid value encountered in multiply
invalid value encountered in multiply


### ITPD naive

In [15]:
naive_accuracies = []
naive_auprs = []
naive_shds = []
naive_total_tests = []
naive_cond_sizes = []
naive_itpd_oracle_times = []

for i in range(iters):
    start = time.time()
    it = ITPDNaive(dfs[i],
                  var_names = var_names,
                  pairs = pairs,
                  independence_test = test,
                  alpha = alpha,
                  tau_max = tau_max,
                  true_adj = adj_matrices[i])
    accuracy, aupr, shd = it.itpd_naive()
    naive_itpd_oracle_times.append(time.time() - start)
    
    naive_total_tests.append(np.sum(it.total_tests))
    naive_cond_sizes_flat = [item for sublist in it.conditioning_set_sizes for item in sublist]
    naive_cond_sizes += naive_cond_sizes_flat
    
    naive_accuracies.append(accuracy)
    naive_auprs.append(aupr)
    naive_shds.append(shd)

# Get 95% confidence intervals.
# Assuming t distribution.
naive_time_ci = st.t.interval(0.95, 
                        len(naive_itpd_oracle_times)-1, 
                        loc = np.mean(naive_itpd_oracle_times), 
                        scale = st.sem(naive_itpd_oracle_times))
naive_tests_ci = st.t.interval(0.95, 
                        len(naive_total_tests)-1, 
                        loc = np.mean(naive_total_tests), 
                        scale = st.sem(naive_total_tests))
naive_cond_ci = st.t.interval(0.95, 
                        len(naive_cond_sizes)-1, 
                        loc = np.mean(naive_cond_sizes), 
                        scale = st.sem(naive_cond_sizes))
naive_auprs_ci = st.t.interval(0.95, 
                        len(naive_auprs)-1, 
                        loc = np.mean(naive_auprs), 
                        scale = st.sem(naive_auprs))
naive_shds_ci = st.t.interval(0.95, 
                        len(naive_shds)-1, 
                        loc = np.mean(naive_shds), 
                        scale = st.sem(naive_shds))

print()
print(f"Mean runtime     : {round(np.mean(naive_itpd_oracle_times),4)}s {ci_to_float(naive_time_ci)}")
print(f"Mean total tests : {round(np.mean(naive_total_tests),2)} {ci_to_float(naive_tests_ci)}")
print(f"Mean cond. set : {round(np.mean(naive_cond_sizes),2)} {ci_to_float(naive_cond_ci)}")
print()
print(f"Mean AUPR        : {round(np.mean(naive_auprs)*100,2)} {ci_to_float(naive_auprs_ci)}")
print(f"Mean SHD         : {round(np.mean(naive_shds),2)} {ci_to_float(naive_shds_ci)}")

ITPD complete in 3.385 seconds.
ITPD complete in 3.1758 seconds.
ITPD complete in 3.3134 seconds.
ITPD complete in 3.3185 seconds.
ITPD complete in 3.5638 seconds.
ITPD complete in 3.4954 seconds.
ITPD complete in 3.2633 seconds.
ITPD complete in 3.3593 seconds.
ITPD complete in 3.6142 seconds.
ITPD complete in 3.3664 seconds.

Mean runtime     : 3.3866s (3.29, 3.484)
Mean total tests : 18380.4 (17996.518, 18764.282)
Mean cond. set : 66.68 (66.403, 66.957)

Mean AUPR        : 100.0 (nan, nan)
Mean SHD         : 0.0 (nan, nan)


invalid value encountered in multiply
invalid value encountered in multiply


In [16]:
np.mean(itpd_oracle_times) / np.mean(naive_itpd_oracle_times)

np.float64(0.8893537550708258)

In [17]:
np.mean(total_tests) / np.mean(naive_total_tests)

np.float64(0.8884681508563469)

In [18]:
np.mean(cond_sizes) / np.mean(naive_cond_sizes)

np.float64(1.0)

# End of document